# 🎯 Experiment 00 — Training

Fine-tunes **Gemma 3n** for multilingual ASR (Lingala + Shona + Luganda) with
LoRA, then **saves the adapter + processor** to `ARTIFACT_DIR`.

This notebook does training only. Run `02_inference.ipynb` afterwards to load
these artifacts and generate a submission.

**Pipeline:** install → config → HF login → load data → load model →
collator → train → **save artifacts**.

## 1. Install dependencies

In [ ]:
%%capture
!pip install -q \
    transformers>=4.52.0 \
    datasets \
    peft \
    trl \
    jiwer \
    timm \
    librosa \
    soundfile \
    accelerate \
    bitsandbytes \
    huggingface_hub \
    torchao \
    pandas

print("✅ Dependencies installed.")

## 2. Configuration

In [ ]:
import dataclasses
from typing import Final

# ---------------------------------------------------------------------------
# Model
# ---------------------------------------------------------------------------
#   "google/gemma-3n-E2B-it"  — ~2B effective params (recommended for T4)
#   "google/gemma-4n-E4B-it"  — ~4B effective params (recommended for A100)
MODEL_ID: Final[str] = "google/gemma-3n-E2B-it"

# ---------------------------------------------------------------------------
# Dataset
# ---------------------------------------------------------------------------
DATASET_ID: Final[str] = "google/WaxalNLP"

# This challenge scores Lingala, Shona and Luganda. We train ONE multilingual
# model on all three, because Phase 2 provides no language metadata — the model
# must transcribe from audio alone.
LANGUAGES: Final[list[str]] = ["lin", "sna", "lug"]

# ---------------------------------------------------------------------------
# Prompts (MUST be identical in train + inference)
# ---------------------------------------------------------------------------
SYSTEM_MESSAGE: Final[str] = (
    "You are an assistant that transcribes speech accurately."
)
USER_MESSAGE: Final[str] = "Please transcribe this audio."

# ---------------------------------------------------------------------------
# Audio
# ---------------------------------------------------------------------------
SAMPLE_RATE: Final[int] = 16_000

# ---------------------------------------------------------------------------
# Artifacts — THE coupling point between the two notebooks.
# Training writes the LoRA adapter + processor here; inference reads from here.
# Point this at persistent storage (Drive / a bucket) so it survives across
# separate Colab sessions.
# ---------------------------------------------------------------------------
ARTIFACT_DIR: Final[str] = "./gemma3n-asr-multilingual"

RANDOM_SEED: Final[int] = 42

# ---------------------------------------------------------------------------
# LoRA
# ---------------------------------------------------------------------------
LORA_R: Final[int] = 8
LORA_ALPHA: Final[int] = 16
LORA_DROPOUT: Final[float] = 0.0
LORA_TARGET_MODULES: Final[list[str]] = ["v_proj", "o_proj"]

# ---------------------------------------------------------------------------
# Training
# ---------------------------------------------------------------------------
PER_DEVICE_TRAIN_BATCH_SIZE: Final[int] = 2
GRADIENT_ACCUMULATION_STEPS: Final[int] = 8
LEARNING_RATE: Final[float] = 1e-3
MAX_STEPS: Final[int] = 500       # Increase for better results (e.g. 3000)
MAX_SEQ_LENGTH: Final[int] = 64
EVAL_STEPS: Final[int] = 100
LOGGING_STEPS: Final[int] = 10
SAVE_STEPS: Final[int] = 100
NUM_VALIDATION_EXAMPLES: Final[int] = 200

print(f"Model     : {MODEL_ID}")
print(f"Languages : {LANGUAGES}")
print(f"Artifacts : {ARTIFACT_DIR}")

## 3. Authenticate with Hugging Face

In [ ]:
from huggingface_hub import login

# Gemma is a gated model — accept its license on the model card, then log in
# with a read token: https://huggingface.co/settings/tokens
login()

## 4. Load & interleave the WaxalNLP data (multilingual)

In [ ]:
from collections.abc import Mapping, Sequence
from typing import Any

import numpy as np
import librosa
import datasets


def audio_to_array(audio: Any) -> np.ndarray:
    """Returns a mono 1-D float32 array from a datasets Audio value.

    Works with both the classic dict form ({'array', 'sampling_rate'}) and the
    torchcodec ``AudioDecoder`` returned by datasets >= 4.
    """
    sr = SAMPLE_RATE
    if isinstance(audio, Mapping):
        arr = np.asarray(audio["array"], dtype=np.float32)
        sr = int(audio.get("sampling_rate", SAMPLE_RATE))
    elif hasattr(audio, "get_all_samples"):          # torchcodec AudioDecoder
        samples = audio.get_all_samples()
        arr = np.asarray(samples.data, dtype=np.float32)
        sr = int(getattr(samples, "sample_rate", SAMPLE_RATE))
    else:
        arr = np.asarray(audio, dtype=np.float32)
    if arr.ndim > 1:                                 # downmix channels to mono
        arr = arr.mean(axis=0)
    arr = arr.flatten()
    if sr != SAMPLE_RATE:
        arr = librosa.resample(arr, orig_sr=sr, target_sr=SAMPLE_RATE)
    return arr


def format_for_chat(
    example: Mapping[str, Any],
    system_message: str = SYSTEM_MESSAGE,
    user_message: str = USER_MESSAGE,
) -> dict[str, Any]:
    """Converts a decoded example into a chat-formatted message list."""
    audio_array = audio_to_array(example["audio"])
    return {
        **example,
        "messages": [
            {"role": "system",
             "content": [{"type": "text", "text": system_message}]},
            {"role": "user",
             "content": [
                 {"type": "audio", "audio": audio_array},
                 {"type": "text", "text": user_message},
             ]},
            {"role": "assistant",
             "content": [{"type": "text", "text": str(example["transcription"])}]},
        ],
    }


def format_batch(batch: dict[str, list[Any]]) -> dict[str, list[Any]]:
    """Formats a batch of decoded examples into chat messages."""
    num = len(batch["transcription"])
    examples = [{k: batch[k][i] for k in batch} for i in range(num)]
    formatted = [format_for_chat(ex) for ex in examples]
    return {k: [ex[k] for ex in formatted] for k in formatted[0]}


def load_split(language: str, split: str) -> datasets.IterableDataset:
    """Loads a single language/split, resampled to SAMPLE_RATE.

    NOTE: it does NOT apply the chat-formatting map. Callers map AFTER any
    ``interleave_datasets`` call, because interleave must infer Arrow features
    and cannot do so from raw/decoded audio objects (torchcodec AudioDecoder).
    """
    ds = datasets.load_dataset(
        DATASET_ID, name=f"{language}_asr", split=split, streaming=True
    )
    return ds.cast_column("audio", datasets.Audio(sampling_rate=SAMPLE_RATE))


print(f"Loading + interleaving languages: {LANGUAGES}")
train_parts = [load_split(lang, "train") for lang in LANGUAGES]
val_parts = [load_split(lang, "validation") for lang in LANGUAGES]

# Interleave the RAW (unmapped) streams so feature inference uses the declared
# schema. Balanced sampling; keep going until every stream is used up.
train_ds = datasets.interleave_datasets(
    train_parts,
    probabilities=[1 / len(LANGUAGES)] * len(LANGUAGES),
    seed=RANDOM_SEED,
    stopping_strategy="all_exhausted",
)
val_ds = datasets.interleave_datasets(val_parts, seed=RANDOM_SEED)

# Apply chat formatting AFTER interleaving.
train_ds = train_ds.map(format_batch, batched=True, batch_size=32)
val_ds = val_ds.map(format_batch, batched=True, batch_size=32)

print("✅ Dataset ready.")
sample = next(iter(train_ds))
print(f"Transcription: {sample['transcription']}")
print(f"Audio shape  : {audio_to_array(sample['audio']).shape}")

## 5. Load model & processor

In [ ]:
import timm  # Required by Gemma3n; import before transformers loads it
import torch
import transformers


def load_model_and_processor(model_id: str):
    """Loads a Gemma3n model + processor from the Hub."""
    processor = transformers.AutoProcessor.from_pretrained(model_id)
    processor.tokenizer.padding_side = "right"
    model = transformers.Gemma3nForConditionalGeneration.from_pretrained(
        model_id, torch_dtype=torch.bfloat16, device_map="auto",
    )
    return model, processor


print(f"Loading {MODEL_ID} …")
model, processor = load_model_and_processor(MODEL_ID)
print(f"✅ Model loaded — device: {model.device}, dtype: {model.dtype}")

## 6. Data collator

In [ ]:
import functools


def _mask_labels(labels: torch.Tensor, proc) -> torch.Tensor:
    """Masks padding/special tokens to -100 so loss ignores them."""
    labels = labels.clone()
    tok = proc.tokenizer
    special_attrs = ["pad_token_id", "image_token_id", "audio_token_id",
                     "boi_token_id", "eoi_token_id"]
    mask_ids = [getattr(tok, a) for a in special_attrs
                if getattr(tok, a, None) is not None]
    if mask_ids:
        labels[torch.isin(labels, torch.tensor(mask_ids, device=labels.device))] = -100
    return labels


def collate_fn(examples, proc):
    """Tokenises a batch of chat examples and builds masked labels."""
    texts = [proc.apply_chat_template(ex["messages"], tokenize=False,
                                      add_generation_prompt=False)
             for ex in examples]
    audios = [audio_to_array(ex["audio"]) for ex in examples]
    batch = proc(text=texts, audio=audios, return_tensors="pt", padding=True)
    batch = {k: (v.detach().clone() if isinstance(v, torch.Tensor) else v)
             for k, v in batch.items()}
    batch["labels"] = _mask_labels(batch["input_ids"], proc)
    return batch


collator = functools.partial(collate_fn, proc=processor)
print("✅ Collator ready.")

## 7. Fine-tune with LoRA (PEFT + TRL SFT)

LoRA adds small trainable rank-decomposition matrices to selected attention
projections, keeping the rest of the model frozen — feasible on a single GPU.

| Hyperparameter | Value | Notes |
|---|---|---|
| `lora_r` | 8 | Increase to 16/32 for capacity |
| `target_modules` | `v_proj, o_proj` | Add `q_proj, k_proj` for more params |
| `max_steps` | 500 | ~3000 recommended for production |

In [ ]:
import peft
import trl


class _SFTTrainer(trl.SFTTrainer):
    """No-op create_model_card to avoid importlib issues."""
    def create_model_card(self, model_name=None, dataset_name=None, tags=None):
        pass


peft_config = peft.LoraConfig(
    task_type="CAUSAL_LM",
    r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
    target_modules=LORA_TARGET_MODULES, bias="none",
    use_rslora=False, use_dora=False,
)

training_args = trl.SFTConfig(
    output_dir=ARTIFACT_DIR,
    max_steps=MAX_STEPS,
    eval_strategy="steps", eval_steps=EVAL_STEPS,
    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    learning_rate=LEARNING_RATE,
    logging_steps=LOGGING_STEPS, save_steps=SAVE_STEPS,
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    report_to="none", run_name="gemma3n-asr-multilingual",
    dataset_kwargs={"skip_prepare_dataset": True},
    remove_unused_columns=False,
    max_length=MAX_SEQ_LENGTH, packing=False,
    dataloader_num_workers=2, seed=RANDOM_SEED,
)

shuffled_train = train_ds.shuffle(buffer_size=1000, seed=RANDOM_SEED).repeat(None)
val_ds_fixed = val_ds.take(NUM_VALIDATION_EXAMPLES)

trainer = _SFTTrainer(
    model=model, args=training_args, data_collator=collator,
    train_dataset=shuffled_train, eval_dataset=val_ds_fixed,
    peft_config=peft_config,
)

print("✅ Trainer initialised. Starting fine-tuning …")
trainer.train()
print("✅ Fine-tuning complete.")

## 8. Save artifacts

Saves the **LoRA adapter** and the **processor** to `ARTIFACT_DIR`. The
inference notebook reloads the base model and applies this adapter — so we only
persist the small adapter, not the full model.

> On Colab, set `ARTIFACT_DIR` to a Drive path (or copy this folder to Drive /
> a bucket) so it survives after the runtime shuts down.

In [ ]:
import os

os.makedirs(ARTIFACT_DIR, exist_ok=True)

# Save the trained LoRA adapter (small) and the processor/tokenizer config.
trainer.save_model(ARTIFACT_DIR)      # writes adapter_config.json + adapter weights
processor.save_pretrained(ARTIFACT_DIR)

print(f"✅ Saved artifacts to: {os.path.abspath(ARTIFACT_DIR)}")
print("Contents:")
for f in sorted(os.listdir(ARTIFACT_DIR)):
    print("  ", f)
print("\nNext: run 02_inference.ipynb with the SAME ARTIFACT_DIR.")